# 02 — Feature Engineering & Construction de la Target

## Objectifs de ce notebook
1. Définir la **target binaire** (réactivation) avec un split temporel propre
2. Construire **3 familles de features** : RFM modifié, expérience 1ère commande, géo
3. Sortir une table d'apprentissage `customer_features.csv` prête pour la modélisation
4. Préparer 2-3 agrégations pour le dashboard Power BI

---

## 🎯 Décision stratégique : Approche A — Prédiction de réactivation

### Le contexte qui force ce choix
L'exploration de l'étape 1 a révélé un fait critique :
- **97% des clients Olist n'ont qu'une seule commande** (90 557 sur 93 358)
- Seulement 2 573 clients ont 2 commandes
- À peine 228 clients ont 3 commandes ou plus

Ce constat **invalide l'approche RFM classique** habituellement utilisée 
sur ce dataset, dans laquelle la composante "Frequency" serait à 1 pour 
quasiment tous les clients (donc inutilisable comme variable discriminante).

### Définition opérationnelle de la target

> Un client est considéré comme **churné (target = 1)** s'il n'a passé 
> aucune nouvelle commande dans les 180 jours suivant la date de coupure T0.

- **T0 = 2018-02-01** (date de coupure)
- **Fenêtre d'observation** : avant T0 → on construit les caracteristique (feature)
- **Fenêtre de prédiction** : T0 → T0+180j → on observe si le client revient

Cette structure de **split temporel** est essentielle : elle évite tout 
**data leakage** (utiliser des informations futures pour prédire le passé), 
faute classique dans les projets ML débutants.

### Structure temporelle du dataset

```
2016-09-01 ── 2017-01-01 ──────────── 2018-02-01 ──────── 2018-08-01
   │              │                        │                    │
   │  Démarrage   │   Fenêtre observation  │  Fenêtre prédiction│
   │   (exclu)    │     (features 13 mois) │     (180 jours)    │
   │              │                        │                    │
   └─ bruit       └─ comportement client   └─ on regarde si le  │
      volumes        capturé ici              client revient    │
      faibles                                                   │
```

## 1. Setup et reconnexion à la base

On reprend la connexion DuckDB persistante créée à l'étape 1. Tous les CSV 
sont déjà chargés en tables, on n'a donc rien à recharger.

On définit aussi les **constantes temporelles** du projet en haut du 
notebook — c'est une bonne pratique : si demain on veut tester une 
fenêtre de 90j ou 120j, on change juste une seule variable.

In [16]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path

# Configuration affichage
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# Reconnexion à la base
con = duckdb.connect("../data/olist.duckdb")

# === CONSTANTES DU PROJET ===
T0 = "2018-02-01"                  # Date de coupure
PREDICTION_WINDOW_DAYS = 180       # Fenêtre de prédiction (jours)
OBSERVATION_START = "2017-01-01"   # Début fenêtre d'observation

# Chemin de sortie
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Date de coupure T0 : {T0}")
print(f"Fenêtre de prédiction : {PREDICTION_WINDOW_DAYS} jours")
print(f"Début observation : {OBSERVATION_START}")

Date de coupure T0 : 2018-02-01
Fenêtre de prédiction : 180 jours
Début observation : 2017-01-01


## 2. Construction de la cohorte éligible

**Règle d'éligibilité** : un client doit avoir passé **au moins une commande 
livrée entre le 2017-01-01 et le 2018-02-01** pour être inclus dans l'analyse.

### Pourquoi cette règle
- Si un client n'a pas commandé **avant T0**, on n'a aucune feature pour lui 
  → impossible de le scorer.
- On exclut la phase de démarrage (sept-déc 2016) car les volumes sont 
  trop faibles et le comportement client n'est pas représentatif.


In [14]:
cohorte_query = f"""
WITH eligible_customers AS (
    SELECT DISTINCT c.customer_unique_id
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '{OBSERVATION_START}'
      AND o.order_purchase_timestamp < '{T0}'
)
SELECT COUNT(*) AS n_eligible_customers FROM eligible_customers
"""

n_eligible = con.execute(cohorte_query).fetchone()[0]
print(f"Cohorte éligible : {n_eligible:,} clients uniques")
print(f"   (clients avec au moins 1 commande livrée entre {OBSERVATION_START} et {T0})")

Cohorte éligible : 48,979 clients uniques
   (clients avec au moins 1 commande livrée entre 2017-01-01 et 2018-02-01)


## 3. Construction de la target binaire

Pour chaque client de la cohorte, on regarde s'il a passé au moins une 
nouvelle commande dans la fenêtre **[T0, T0+180j]**.

### Logique
- `target = 1` (churn) → **aucune** commande dans la fenêtre de prédiction
- `target = 0` (actif) → au moins une commande dans la fenêtre

### Détail technique important
On utilise un `LEFT JOIN` sur les commandes futures, puis on compte. Si le 
count = 0, le client a churné. Cette approche est plus robuste qu'un 
`NOT EXISTS` car elle gère naturellement les clients sans aucune commande 
future.

### Ce qu'on doit observer
On s'attend à un **fort déséquilibre de classes** : autour de 95-97% de 
churn (= 1) et seulement 3-5% de réactivations (= 0). On confirmera ça 
en sortie de cellule.

In [7]:
target_query = f"""
WITH eligible_customers AS (
    SELECT DISTINCT c.customer_unique_id
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '{OBSERVATION_START}'
      AND o.order_purchase_timestamp < '{T0}'
),
future_orders AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS n_orders_future
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '{T0}'
      AND o.order_purchase_timestamp < DATE_ADD(DATE '{T0}', INTERVAL {PREDICTION_WINDOW_DAYS} DAY)
    GROUP BY 1
)
SELECT 
    e.customer_unique_id,
    COALESCE(f.n_orders_future, 0) AS n_orders_future,
    CASE WHEN COALESCE(f.n_orders_future, 0) = 0 THEN 1 ELSE 0 END AS churn
FROM eligible_customers e
LEFT JOIN future_orders f ON e.customer_unique_id = f.customer_unique_id
"""

# Persistance en table pour réutilisation dans les CTE suivants
con.execute(f"CREATE OR REPLACE TABLE customer_target AS {target_query}")

# Vérification du déséquilibre de classes
target_dist = con.execute("""
    SELECT 
        churn,
        COUNT(*) AS n_customers,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM customer_target
    GROUP BY churn
    ORDER BY churn
""").df()

print("Distribution de la target :")
print(target_dist)
print(f"\nDéséquilibre attendu — à gérer dans la modélisation (class_weight, SMOTE, threshold tuning)")

Distribution de la target :
   churn  n_customers    pct
0      0          576   1.18
1      1        48403  98.82

Déséquilibre attendu — à gérer dans la modélisation (class_weight, SMOTE, threshold tuning)


## 4. Famille 1 — Features RFM modifié

On construit les 3 indicateurs classiques **avant T0** uniquement (pour 
éviter le data leakage).

### Définitions
- **Recency** : nombre de jours entre la dernière commande et T0. Plus c'est 
  élevé, plus le client est "froid".
- **Frequency** : nombre de commandes livrées avant T0. Sera ≈ 1 pour 97% 
  des clients, mais reste discriminante pour les 3% restants.
- **Monetary** : montant total dépensé avant T0 (somme des `payment_value`).

### Pourquoi calculer ça malgré la frequency dégénérée
Même si la majorité des clients sont mono-acheteurs, **la recency reste 
extrêmement discriminante** : un client qui a commandé il y a 30 jours a 
beaucoup plus de chances de revenir qu'un client qui a commandé il y a 300 jours.

Le **Monetary** est aussi utile : panier élevé = client probablement plus 
fidèle (ou cadeau ponctuel — le modèle décidera).

In [13]:
rfm_query = f"""
WITH customer_orders AS (
    SELECT 
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp < '{T0}'
),
rfm AS (
    SELECT 
        co.customer_unique_id,
        DATE_DIFF('day', MAX(co.order_purchase_timestamp), DATE '{T0}') AS recency_days,
        COUNT(DISTINCT co.order_id) AS frequency,
        SUM(p.payment_value) AS monetary_total,
        AVG(p.payment_value) AS monetary_avg_order
    FROM customer_orders co
    LEFT JOIN order_payments p ON co.order_id = p.order_id
    GROUP BY 1
)
SELECT * FROM rfm
"""

con.execute(f"CREATE OR REPLACE TABLE features_rfm AS {rfm_query}")

# Aperçu et stats descriptives
rfm_df = con.execute("SELECT * FROM features_rfm").df()
print(f"Features RFM calculées sur {len(rfm_df):,} clients\n")
print(rfm_df[['recency_days', 'frequency', 'monetary_total', 'monetary_avg_order']].describe())

Features RFM calculées sur 49,237 clients

       recency_days     frequency  monetary_total  monetary_avg_order
count  49237.000000  49237.000000    49236.000000        49236.000000
mean     148.522310      1.031013      163.459530          156.106366
std      105.745536      0.196010      227.173238          217.664771
min        1.000000      1.000000       10.070000            1.856818
25%       62.000000      1.000000       62.780000           60.180000
50%      129.000000      1.000000      106.160000          102.030000
75%      230.000000      1.000000      180.535000          172.795000
max      504.000000      8.000000    13664.080000        13664.080000


## 5. Famille 2 — Features expérience 1ère commande

**C'est la famille la plus stratégique** pour ce projet. Puisque 97% des 
clients n'ont qu'une commande, leur **expérience sur cette commande** est 
le meilleur prédicteur de leur retour.

### Hypothèses business à valider
1. Une **livraison en retard** ↘ probabilité de retour
2. Une **mauvaise note review** ↘ probabilité de retour
3. Un **paiement par boleto** (non récurrent au Brésil) ↗ probabilité de churn
4. Un **panier élevé sur 1 seul article** = achat opportuniste → churn probable

### Features construites
- `delivery_days` : durée réelle de livraison (commande → réception)
- `delivery_delay_vs_estimate` : retard ou avance vs date estimée (en jours)
- `is_late` : flag livraison en retard
- `review_score` : note moyenne des reviews
- `payment_type_main` : mode de paiement principal
- `payment_installments` : nombre de mensualités
- `n_items` : nombre d'articles dans la 1ère commande
- `n_distinct_categories` : diversité produit dans la 1ère commande

### Note méthodologique
On utilise la **première** commande car elle existe pour 100% des clients. 
On pourrait calculer les mêmes features sur la "dernière" commande pour 
les multi-acheteurs, mais on garde l'exercice simple ici.

In [12]:
first_order_query = f"""
WITH first_order AS (
    SELECT 
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id 
            ORDER BY o.order_purchase_timestamp
        ) AS rn
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp < '{T0}'
),
first_order_filtered AS (
    SELECT * FROM first_order WHERE rn = 1
),
items_agg AS (
    SELECT 
        order_id,
        COUNT(*) AS n_items,
        COUNT(DISTINCT product_id) AS n_distinct_products,
        SUM(price) AS total_price,
        SUM(freight_value) AS total_freight
    FROM order_items
    GROUP BY 1
),
categories_agg AS (
    SELECT 
        oi.order_id,
        COUNT(DISTINCT p.product_category_name) AS n_distinct_categories
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY 1
),
payment_agg AS (
    SELECT 
        order_id,
        MAX(payment_type) AS payment_type_main,
        MAX(payment_installments) AS payment_installments_max,
        SUM(payment_value) AS payment_total
    FROM order_payments
    GROUP BY 1
),
review_agg AS (
    SELECT 
        order_id,
        AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY 1
)
SELECT 
    fof.customer_unique_id,
    DATE_DIFF('day', fof.order_purchase_timestamp, fof.order_delivered_customer_date) AS delivery_days,
    DATE_DIFF('day', fof.order_delivered_customer_date, fof.order_estimated_delivery_date) AS delivery_delay_vs_estimate,
    CASE WHEN fof.order_delivered_customer_date > fof.order_estimated_delivery_date THEN 1 ELSE 0 END AS is_late,
    rev.review_score,
    pay.payment_type_main,
    pay.payment_installments_max,
    pay.payment_total,
    ia.n_items,
    ia.n_distinct_products,
    cat.n_distinct_categories,
    ia.total_price,
    ia.total_freight,
    CASE WHEN ia.total_price > 0 THEN ia.total_freight / ia.total_price ELSE NULL END AS freight_ratio
FROM first_order_filtered fof
LEFT JOIN items_agg ia ON fof.order_id = ia.order_id
LEFT JOIN categories_agg cat ON fof.order_id = cat.order_id
LEFT JOIN payment_agg pay ON fof.order_id = pay.order_id
LEFT JOIN review_agg rev ON fof.order_id = rev.order_id
"""

con.execute(f"CREATE OR REPLACE TABLE features_first_order AS {first_order_query}")

first_order_df = con.execute("SELECT * FROM features_first_order").df()
print(f"Features 1ère commande calculées sur {len(first_order_df):,} clients\n")
print(first_order_df.describe(include='all').T.head(20))

Features 1ère commande calculées sur 49,237 clients

                              count unique                               top  \
customer_unique_id            49237  49237  10e49a12540465d3141f898aeee0a6b4   
delivery_days               49235.0   <NA>                              <NA>   
delivery_delay_vs_estimate  49235.0   <NA>                              <NA>   
is_late                     49237.0    NaN                               NaN   
review_score                48866.0    NaN                               NaN   
payment_type_main             49236      4                       credit_card   
payment_installments_max    49236.0   <NA>                              <NA>   
payment_total               49236.0    NaN                               NaN   
n_items                     49237.0    NaN                               NaN   
n_distinct_products         49237.0    NaN                               NaN   
n_distinct_categories       49237.0    NaN                         

## 6. Famille 3 — Features géographiques

L'État brésilien du client peut être prédictif :
- États du Sud-Est (SP, RJ, MG) : marché mature, infrastructures logistiques 
  meilleures → satisfaction client souvent supérieure → moins de churn ?
- États du Nord/Nord-Est : délais de livraison plus longs, infrastructures 
  moins développées → potentiellement plus de churn

### Note d'encodage
On garde l'État en variable catégorielle. Pour la modélisation (étape 3), 
on choisira entre :
- **One-hot encoding** : ajoute 27 colonnes mais reste lisible
- **Target encoding** : plus compact mais risque de leakage si mal fait

On ajoute aussi une feature plus fine : la **région** (5 régions au Brésil), 
plus interprétable côté Power BI.

In [11]:
geo_query = f"""
WITH eligible_customers AS (
    SELECT DISTINCT c.customer_unique_id
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '{OBSERVATION_START}'
      AND o.order_purchase_timestamp < '{T0}'
),
customer_state AS (
    -- On prend l'état le plus fréquent par client (au cas où il aurait déménagé)
    SELECT 
        c.customer_unique_id,
        c.customer_state,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id 
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM customers c
    WHERE c.customer_unique_id IN (SELECT customer_unique_id FROM eligible_customers)
    GROUP BY 1, 2
)
SELECT 
    customer_unique_id,
    customer_state,
    CASE 
        WHEN customer_state IN ('SP', 'RJ', 'MG', 'ES') THEN 'Sudeste'
        WHEN customer_state IN ('PR', 'SC', 'RS') THEN 'Sul'
        WHEN customer_state IN ('GO', 'MT', 'MS', 'DF') THEN 'Centro-Oeste'
        WHEN customer_state IN ('BA', 'SE', 'AL', 'PE', 'PB', 'RN', 'CE', 'PI', 'MA') THEN 'Nordeste'
        WHEN customer_state IN ('AM', 'PA', 'AC', 'RO', 'RR', 'AP', 'TO') THEN 'Norte'
        ELSE 'Other'
    END AS customer_region
FROM customer_state
WHERE rn = 1
"""

con.execute(f"CREATE OR REPLACE TABLE features_geo AS {geo_query}")

geo_df = con.execute("SELECT * FROM features_geo").df()
print(f" Features géo calculées sur {len(geo_df):,} clients\n")
print("Répartition par région :")
print(geo_df['customer_region'].value_counts(normalize=True).round(3))

 Features géo calculées sur 48,979 clients

Répartition par région :
customer_region
Sudeste         0.674
Sul             0.148
Nordeste        0.099
Centro-Oeste    0.058
Norte           0.021
Name: proportion, dtype: float64


## 7. Assemblage de la table d'apprentissage finale

On joint les 4 tables construites :
- `customer_target` (la cible)
- `features_rfm` (RFM modifié)
- `features_first_order` (expérience 1ère commande)
- `features_geo` (géographie)

### Choix méthodologique : `LEFT JOIN` depuis la target
On part de la cohorte définie dans `customer_target` et on enrichit. Tous 
les clients de la cohorte doivent se retrouver dans la table finale.

### Vérification post-jointure
On doit retrouver le **même nombre de lignes** que dans `customer_target`. 
Si on en perd, c'est qu'il y a un souci dans une table de features.

In [18]:
final_query = """
SELECT 
    t.customer_unique_id,
    t.churn,
    t.n_orders_future,
    -- RFM
    rfm.recency_days,
    rfm.frequency,
    rfm.monetary_total,
    rfm.monetary_avg_order,
    -- 1ère commande
    fo.delivery_days,
    fo.delivery_delay_vs_estimate,
    fo.is_late,
    fo.review_score,
    fo.payment_type_main,
    fo.payment_installments_max,
    fo.n_items,
    fo.n_distinct_products,
    fo.n_distinct_categories,
    fo.total_price,
    fo.total_freight,
    fo.freight_ratio,
    -- Géo
    geo.customer_state,
    geo.customer_region
FROM customer_target t
LEFT JOIN features_rfm rfm ON t.customer_unique_id = rfm.customer_unique_id
LEFT JOIN features_first_order fo ON t.customer_unique_id = fo.customer_unique_id
LEFT JOIN features_geo geo ON t.customer_unique_id = geo.customer_unique_id
"""

df_final = con.execute(final_query).df()

print(f"Table finale : {len(df_final):,} lignes × {df_final.shape[1]} colonnes")
print(f"\nVérification : nombre de lignes = nombre de clients dans la cohorte ?")
print(f"   Cohorte: {con.execute('SELECT COUNT(*) FROM customer_target').fetchone()[0]:,}")
print(f"   Final  : {len(df_final):,}")

Table finale : 48,979 lignes × 21 colonnes

Vérification : nombre de lignes = nombre de clients dans la cohorte ?
   Cohorte: 48,979
   Final  : 48,979
